<a href="https://colab.research.google.com/github/UZUddin/PitLane-Explained/blob/main/pitlane_explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PitLane Explained— IBM SkillsBuild AI Builders Challenge

In [2]:
# PitLane Explained — IBM SkillsBuild AI Builders Challenge
# AI Race Companion for Casual F1 Fans
# Tools: IBM Granite, Docling, LangChain, ChromaDB

In [3]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    "transformers==4.49.0" \
    pillow \
    langchain_classic \
    langchain_core \
    langchain_huggingface sentence_transformers \
    langchain_chroma chromadb \
    docling \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git"
! echo "::endgroup::"

::group::Install Dependencies
Using Python 3.12.13 environment at: /usr
Resolved 162 packages in 719ms
Checked 162 packages in 4ms
::endgroup::


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(model_name=embeddings_model_path)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

print("✅ Embeddings model ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Embeddings model ready!


In [5]:
import os
from google.colab import userdata
from ibm_granite_community.notebook_utils import get_env_var
from langchain_replicate import ChatReplicate

os.environ["REPLICATE_API_TOKEN"] = userdata.get('REPLICATE_API_TOKEN')

model_path = "ibm-granite/granite-4.1-8b"
model = ChatReplicate(
    model=model_path,
    replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
    model_kwargs={
        "max_completion_tokens": 1000,
        "min_tokens": 100,
    },
)

print("✅ Granite model ready!")

✅ Granite model ready!


In [9]:
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.types.doc.document import TableItem
from docling_core.types.doc.labels import DocItemLabel
from langchain_core.documents import Document
import itertools

converter = DocumentConverter()

# Source 1: Wikipedia race page (narrative, works reliably with Docling)
sources = [
    "https://en.wikipedia.org/wiki/2024_Monaco_Grand_Prix",
]

conversions = {
    source.split("/")[-1]: converter.convert(source=source).document
    for source in sources
}

# Process text chunks
doc_id = 0
texts = []
for source, docling_document in conversions.items():
    for chunk in HybridChunker(tokenizer=embeddings_tokenizer).chunk(docling_document):
        items = chunk.meta.doc_items
        if len(items) == 1 and isinstance(items[0], TableItem):
            continue
        refs = " ".join(map(lambda item: item.get_ref().cref, items))
        text = chunk.text
        document = Document(
            page_content=text,
            metadata={"doc_id": (doc_id := doc_id + 1), "source": source, "ref": refs},
        )
        texts.append(document)

# Process tables
tables = []
for source, docling_document in conversions.items():
    for table in docling_document.tables:
        if table.label in [DocItemLabel.TABLE]:
            ref = table.get_ref().cref
            text = table.export_to_markdown(docling_document)
            document = Document(
                page_content=text,
                metadata={"doc_id": (doc_id := doc_id + 1), "source": source, "ref": ref},
            )
            tables.append(document)

documents = list(itertools.chain(texts, tables))
print(f"✅ {len(texts)} text chunks + {len(tables)} tables from Wikipedia via Docling")

✅ 17 text chunks + 21 tables from Wikipedia via Docling


In [10]:
from docling.document_converter import DocumentConverter
from urllib.parse import urlparse

converter = DocumentConverter()

sources = [
    "https://www.fia.com/system/files/documents/fia_2026_f1_regulations_-_section_b_sporting_-_iss_06_-_2026-04-28.pdf",
]

conversions = {
    source.split("/")[-1]: converter.convert(source=source).document
    for source in sources
}

print("✅ 2024 Monaco Grand Prix page loaded!")

[INFO] 2026-05-25 01:33:40,959 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-25 01:33:40,995 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-25 01:33:40,998 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-25 01:33:41,169 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-25 01:33:41,176 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-25 01:33:41,178 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-25 01:33:41,249 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-25 01:33:41,328 [RapidOCR] download_file.py:60: File exists and is valid: /usr

✅ 2024 Monaco Grand Prix page loaded!


In [11]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.types.doc.document import TableItem
from docling_core.types.doc.labels import DocItemLabel
from langchain_core.documents import Document
import itertools

# Process text chunks
doc_id = 0
texts = []
for source, docling_document in conversions.items():
    for chunk in HybridChunker(tokenizer=embeddings_tokenizer).chunk(docling_document):
        items = chunk.meta.doc_items
        if len(items) == 1 and isinstance(items[0], TableItem):
            continue
        text = chunk.text
        document = Document(
            page_content=text,
            metadata={"doc_id": (doc_id := doc_id + 1), "source": source},
        )
        texts.append(document)

# Process tables
tables = []
for source, docling_document in conversions.items():
    for table in docling_document.tables:
        if table.label in [DocItemLabel.TABLE]:
            text = table.export_to_markdown(docling_document)
            document = Document(
                page_content=text,
                metadata={"doc_id": (doc_id := doc_id + 1), "source": source},
            )
            tables.append(document)

documents = list(itertools.chain(texts, tables))
print(f"✅ {len(texts)} text chunks + {len(tables)} tables created")

✅ 227 text chunks + 2 tables created


In [12]:
from langchain_chroma import Chroma

vector_db = Chroma(embedding_function=embeddings_model)
ids = []
for doc in documents:
    ids += vector_db.add_documents([doc])

print(f"✅ {len(ids)} chunks added to vector database")

✅ 229 chunks added to vector database


In [13]:
from ibm_granite_community.langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("{input}")

combine_docs_chain = create_stuff_documents_chain(
    llm=model,
    prompt=prompt_template,
)
rag_chain = create_retrieval_chain(
    retriever=vector_db.as_retriever(),
    combine_docs_chain=combine_docs_chain,
)

print("✅ RAG chain ready!")

✅ RAG chain ready!


In [15]:
import time

def safe_invoke(chain, input_dict, retries=3, wait=15):
    """Invoke a chain with automatic retry on rate limit errors"""
    for attempt in range(retries):
        try:
            return chain.invoke(input_dict)
        except Exception as e:
            if "429" in str(e) or "throttled" in str(e):
                print(f"⏳ Rate limit hit. Waiting {wait} seconds... (attempt {attempt+1}/{retries})")
                time.sleep(wait)
            else:
                raise e
    raise Exception("Max retries exceeded")

print("✅ Rate limit handler ready!")

✅ Rate limit handler ready!


In [16]:
# Feature 1 - Q&A Chatbot
question = "What happens when a safety car is deployed?"
output = safe_invoke(rag_chain, {"input": question})
print("🏎️ PitLane Explained Answer:")
print(output['answer'])

🏎️ PitLane Explained Answer:
When a Safety Car is deployed in Formula 1 according to the 2026 regulations, the following actions occur:

1. **Notification**: The message 'SAFETY CAR DEPLOYED' is sent to all Competitors, FIA light panels display 'SC', and marshal's posts display the Safety Car indicator.

2. **Driver Conduct**:
   - No F1 Car may be driven unnecessarily slowly, erratically, or in a potentially dangerous manner.
   - All F1 Cars must reduce speed and form up in a queue behind the Safety Car, maintaining a maximum allowable gap of ten (10) car lengths apart, unless conditions dictate a larger gap (up to twenty car lengths in poor visibility).

3. **Speed Enforcement**: Drivers must stay above a minimum time set by the FIA ECU at least once in each marshalling sector and at both the first and second safety car lines. Failure to do so can result in penalties ranging from a 5-Second Penalty to a Stop-and-Go Penalty.

4. **Overtaking Restrictions**: Generally, no overtaking i

In [17]:
# Feature 2 - Race Summary
summary_question = """
Give me a short, engaging 3-paragraph summary written for someone
who has never watched F1 before. Explain what a typical Monaco Grand
Prix is like, why it's exciting, and what makes it special.
Use simple language and avoid technical jargon.
"""
summary_output = safe_invoke(rag_chain, {"input": summary_question})
print("🏁 RACE STORY:")
print("=" * 50)
print(summary_output['answer'])

⏳ Rate limit hit. Waiting 15 seconds... (attempt 1/3)
🏁 RACE STORY:
The Monaco Grand Prix is one of the most prestigious and thrilling events in the Formula 1 calendar, held annually on the streets of Monte Carlo, Monaco. Unlike most F1 races that take place on purpose-built circuits, the Monaco Grand Prix winds through the narrow, twisty, and historic streets of the tiny principality, making it a unique and challenging test for both drivers and their cars. The race is known for its glamour, with celebrities, royalty, and fashion icons lining the grandstands, adding to the spectacle.

What makes the Monaco Grand Prix so exciting is the intense competition and the sheer difficulty of navigating the street circuit. Drivers must master a series of tight corners, hairpin turns, and narrow passages, often just inches away from the barriers. The close proximity of the cars and the limited overtaking opportunities mean that every lap is a battle for position, and a single mistake can be costl